
# RAGAS Evaluation & a CRAG-Style Evaluator

**Day 3 — RAG & Agents · Practical 3 of 6 · Companion to the "Advanced & Self-Correcting RAG"
deck**

> **Running in Google Colab:** works on the default **CPU runtime** — this notebook calls an
> LLM API, no local model inference.

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Run RAGAS's core metrics (Faithfulness, Answer Relevancy, Context Precision) against a RAG
   pipeline's outputs
2. Implement a simplified Corrective RAG (CRAG) style retrieval evaluator that grades retrieved
   chunks and routes accordingly
3. See the difference between MEASURING RAG quality (RAGAS) and SELF-CORRECTING it at runtime
   (CRAG)

## Why This Matters for a Law Firm

A RAG system that "seems to work" in a demo isn't the same as one with measured, monitored
quality. This notebook builds both halves: a scoring pass you'd run continuously in CI, and a
runtime self-correction mechanism that catches bad retrievals before they reach a user.

## Notebook Workflow

```mermaid
flowchart TD
    A["RAG pipeline\n(from Notebook 1)"] --> B["Run queries,\ncollect Q/A/context triples"]
    B --> C["RAGAS metrics:\nFaithfulness, Relevancy, Precision"]
    B --> D["CRAG evaluator:\ngrade each retrieved chunk"]
    D --> E["Correct -> use"]
    D --> F["Incorrect -> discard,\nfallback"]
    D --> G["Ambiguous -> combine"]



## Section 1 — Setup


In [ ]:

%pip install -q ragas datasets openai sentence-transformers chromadb

import os
import json
from openai import OpenAI

def get_api_key(env_var_name):
    try:
        from google.colab import userdata
        key = userdata.get(env_var_name)
        if key:
            return key
    except ImportError:
        pass
    return os.environ.get(env_var_name)

OPENAI_API_KEY = get_api_key("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("Add an OPENAI_API_KEY secret in Colab (key icon, left sidebar).")

client = OpenAI(api_key=OPENAI_API_KEY)
print("Client configured.")



## Section 2 — Rebuild the RAG Pipeline

A compact version of Notebook 1's pipeline, so this notebook is self-contained.


In [ ]:

import chromadb
from sentence_transformers import SentenceTransformer

legal_documents = [
    "Section 7.1 Indemnification. The Contractor shall indemnify, defend, and hold harmless the Client from any claims arising from gross negligence or willful misconduct.",
    "Section 9.2 Termination for Convenience. Either party may terminate this Agreement without cause upon sixty (60) days' prior written notice.",
    "Section 12.1 Confidentiality. The Receiving Party shall maintain all Confidential Information in strict confidence and shall not disclose it to any third party.",
    "Section 14.3 Limitation of Liability. In no event shall either party's total liability exceed the total fees paid in the twelve (12) months preceding the claim.",
    "Section 16.1 Governing Law. This Agreement shall be governed by the laws of the State of Delaware, without regard to conflict of laws principles.",
]

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="legal_clauses_eval")
collection.add(
    ids=[f"doc{i}" for i in range(len(legal_documents))],
    embeddings=embedding_model.encode(legal_documents).tolist(),
    documents=legal_documents,
)

def retrieve(query, k=2):
    query_embedding = embedding_model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=k)
    return results["documents"][0]

def answer_with_rag(query, k=2):
    contexts = retrieve(query, k=k)
    context_str = "\n".join(contexts)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer using only the provided context."},
            {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion: {query}"},
        ],
        temperature=0,
    )
    return response.choices[0].message.content, contexts

print("RAG pipeline rebuilt.")



## Section 3 — Collect Evaluation Data

Run a small set of test questions through the pipeline, collecting question/answer/context
triples -- exactly the format RAGAS expects.


In [ ]:

eval_questions = [
    "What happens if the contractor is grossly negligent?",
    "How much notice is required to terminate the agreement?",
    "What is the cap on liability?",
]

# ragas>=0.2 schema: user_input / response / retrieved_contexts (not question/answer/contexts)
eval_records = {"user_input": [], "response": [], "retrieved_contexts": []}

for q in eval_questions:
    answer, contexts = answer_with_rag(q)
    eval_records["user_input"].append(q)
    eval_records["response"].append(answer)
    eval_records["retrieved_contexts"].append(contexts)
    print(f"Q: {q}\nA: {answer}\n")



## Section 4 — Run RAGAS

RAGAS's Faithfulness, Answer Relevancy, and Context Precision metrics, computed automatically
via an LLM-as-judge approach -- no hand-labeled ground truth required for these three.


In [ ]:

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

eval_dataset = Dataset.from_dict(eval_records)

ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(api_key=OPENAI_API_KEY))

results = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

print(results.to_pandas()[["user_input", "faithfulness", "answer_relevancy", "context_precision"]])



**Reading these numbers:** compare each row against the deck's production targets --
Faithfulness ≥0.9, Answer Relevancy ≥0.85, Context Precision ≥0.8. Any row scoring below target
is a concrete, specific signal of where this pipeline needs work (a chunking fix, a better
prompt, more context retrieved) -- not a vague "the answers seem okay" judgment.



## Section 5 — A CRAG-Style Retrieval Evaluator

Now the runtime self-correction half: instead of measuring quality after the fact, grade each
retrieved chunk BEFORE generation, and route based on that grade -- the deck's Correct /
Ambiguous / Incorrect pattern, simplified.


In [ ]:

def crag_grade_chunk(query, chunk):
    # Ask an LLM to grade a single retrieved chunk's relevance to the query.
    # Returns one of: "Correct", "Ambiguous", "Incorrect"
    grading_prompt = (
        f"Query: {query}\n"
        f"Retrieved text: {chunk}\n\n"
        "Grade how relevant this retrieved text is to answering the query. "
        "Respond with exactly one word: Correct, Ambiguous, or Incorrect."
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": grading_prompt}],
        temperature=0,
    )
    grade = response.choices[0].message.content.strip()
    return grade if grade in ("Correct", "Ambiguous", "Incorrect") else "Ambiguous"


def crag_answer(query, k=2):
    contexts = retrieve(query, k=k)
    grades = [crag_grade_chunk(query, c) for c in contexts]

    correct_chunks = [c for c, g in zip(contexts, grades) if g == "Correct"]
    ambiguous_chunks = [c for c, g in zip(contexts, grades) if g == "Ambiguous"]

    if correct_chunks:
        # At least one solidly relevant chunk -- use it (optionally with ambiguous ones too)
        usable_context = correct_chunks + ambiguous_chunks
        routing_decision = "Correct chunk(s) found -- using retrieved context"
    elif ambiguous_chunks:
        usable_context = ambiguous_chunks
        routing_decision = "Only Ambiguous chunk(s) -- using with caution, no fallback available in this demo"
    else:
        usable_context = []
        routing_decision = "All chunks graded Incorrect -- would fall back to web search in a full CRAG system"

    if usable_context:
        context_str = "\n".join(usable_context)
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Answer using only the provided context."},
                {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion: {query}"},
            ],
            temperature=0,
        )
        answer = response.choices[0].message.content
    else:
        answer = "No sufficiently relevant context was found for this question."

    return answer, grades, routing_decision



## Section 6 — Try the CRAG Evaluator

Compare a question our corpus can answer well against one it can't -- watch the grading and
routing decision differ.


In [ ]:

for q in [
    "What happens if the contractor is grossly negligent?",
    "What is the penalty for late payment under this agreement?",  # not in our small corpus
]:
    answer, grades, routing = crag_answer(q)
    print(f"Q: {q}")
    print(f"Chunk grades: {grades}")
    print(f"Routing decision: {routing}")
    print(f"A: {answer}\n")
    print("-" * 70)



## Key Takeaways

1. **RAGAS scores are diagnostic, not just a vague quality feeling** -- Section 4's table names
   specifically which metric is weak, on which question, informing exactly what to fix.
2. **CRAG's grading step catches a bad retrieval BEFORE generation**, rather than measuring
   quality only after the fact -- Section 6's second query shows the evaluator correctly
   recognizing when the corpus genuinely doesn't have the answer, rather than letting the model
   hallucinate a confident-sounding but ungrounded response.
3. **These two techniques are complementary, as the deck states**: RAGAS tells you where your
   pipeline is systematically weak (informing what to fix); CRAG catches individual bad
   retrievals live, at query time (informing what to do right now).

**Next up:** the *ColPali* notebook — retrieving from document images directly, no OCR.
